# Trazo Pipeline Walkthrough

This notebook walks through the full 5-step **trazo** pipeline for FTW-style agricultural field boundary detection.

| Step | Module | Description |
|------|--------|-------------|
| 1 | `pt1_createdata` | Create training grid + download Sentinel-2 chips |
| 2 | `pt2_dataprep` | Resize, mask, pair, and export chips |
| 3 | `pt3_finetune` | Fine-tune an existing checkpoint |
| 4 | `pt4_train` | Train a new model from scratch |
| 5 | `pt5_inference` | Run inference on new imagery |

---

## Installation

Install the package and the extras for whichever steps you need:

In [ ]:
# Install the package. Steps 1 and 2 run on Python 3.10-3.12;
# Steps 3, 4 and 5 need 3.11 or 3.12 (ftw-tools requires >=3.11,<3.13).
# !pip install -e ".[pt1,pt2]"    # steps 1 and 2
# !pip install -e ".[pt3]"        # step 3 fine-tuning and merging
# !pip install -e ".[pt4]"        # step 4 training
# !pip install -e ".[pt5]"        # step 5 inference
# !pip install -e ".[active]"     # step 3.2 active-learning sampling
# !pip install -e ".[all]"        # everything

---
## Step 0a - Verify the install

If this cell fails, nothing further in the notebook will work. It runs the real
pipeline code on synthetic fields and chips, so a pass means the install is wired
up correctly - not that your own data is valid. No network access is used.

In [ ]:
# Check the install before touching any real data.
# Runs Step 1.1 gridding and the whole Step 2 chain over synthetic inputs.
from trazo.smoke import run

run()   # raises on failure, prints PASS on success

---
## Step 0 — Set your paths

Edit these variables once; the rest of the notebook uses them.

In [ ]:
from pathlib import Path

# Root workspace — all outputs go under here
WORKSPACE = Path("./trazo_workspace").resolve()

# Input field boundary file (shp / geojson / gpkg / parquet)
FIELDS_FILE = Path("./my_fields.geojson").resolve()

# Pretrained checkpoint (needed for pt3 / pt4 fine-tuning / pt5)
CHECKPOINT = Path("./model.ckpt").resolve()

# Training config YAML (pt4)
TRAIN_CONFIG = Path("./configs/train_config.yaml").resolve()

# Create subdirectories
dirs = {
    "grid":      WORKSPACE / "01_grid",
    "chips":     WORKSPACE / "02_chips",
    "dataprep":  WORKSPACE / "03_dataprep",
    "checkpoints": WORKSPACE / "04_checkpoints",
    "inference": WORKSPACE / "05_inference",
}
for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

print("Workspace:", WORKSPACE)
for name, path in dirs.items():
    print(f"  {name}: {path}")

---
## Step 1a — Create a Training Grid

Takes your field boundary polygons and lays a 2560 m grid over them, producing one cell per potential training chip.

**CLI equivalent:**
```bash
trazo-pt1-create grid \
  --input my_fields.geojson \
  --output-dir ./trazo_workspace/01_grid \
  --cell-size-meters 2560 \
  --output-format geojson
```

In [ ]:
import sys

sys.argv = [
    "trazo-pt1-create",
    "--input",            str(FIELDS_FILE),
    "--output-dir",       str(dirs["grid"]),
    "--cell-size-meters", "2560",
    "--output-format",    "geojson",
]

from trazo.pt1_createdata.gridding import main as gridding_main
gridding_main()

In [ ]:
# Inspect the grid output
import geopandas as gpd
import glob

grid_files = glob.glob(str(dirs["grid"] / "*_grid.geojson"))
print("Grid files:", grid_files)

if grid_files:
    gdf = gpd.read_file(grid_files[0])
    print(f"\n{len(gdf)} grid cells | CRS: {gdf.crs}")
    # Step 1.1 writes chip_id, chip_area, cov_area, cov_pct - all 10 characters
    # or shorter, so shapefile output keeps the same names.
    print(gdf[["chip_id", "chip_area", "cov_pct"]].head())
    gdf.plot(column="cov_pct", legend=True, figsize=(8, 6))

---
## Step 1b — Download Sentinel-2 Chips  *(requires `pt1` extras)*

Downloads Sentinel-2 imagery from Planetary Computer for each grid cell across planting and harvest windows.

**CLI equivalent:**
```bash
trazo-pt1-create chips \
  --grid      ./trazo_workspace/01_grid/my_fields_grid.geojson \
  --out-dir   ./trazo_workspace/02_chips \
  --start     2023-06-01 \
  --end       2023-09-30
```

In [ ]:
# Find the grid file created above
grid_geojson = sorted(dirs["grid"].glob("*_grid.geojson"))
if not grid_geojson:
    raise FileNotFoundError("No grid geojson found - run Step 1a first.")
grid_path = grid_geojson[0]
print("Using grid:", grid_path)

# Download planting and harvest chips.
# --chipid-field defaults to chip_id, the column Step 1.1 just wrote, so the
# two steps line up with no extra flags.
sys.argv = [
    "trazo-pt1-create",
    "--input",         str(grid_path),
    "--year-constant", "2023",
    "--chip-size",     "256",
    "--batch-size",    "10",
]

# Run `trazo-pt1-create chips --help` for the output-directory and cloud flags.
from trazo.pt1_createdata import plantingharvest
plantingharvest.main()

---
## Step 2a — Pair Window A + B into 8-Band Stacks

Combines planting-season (window A) and harvest-season (window B) 4-band TIFFs into single 8-band chips.

**CLI equivalent:**
```bash
trazo-pt2-dataprep pair-stacks \
  --window-a-dir ./trazo_workspace/02_chips/window_a \
  --window-b-dir ./trazo_workspace/02_chips/window_b \
  --out-dir      ./trazo_workspace/03_dataprep/stacked
```

In [ ]:
stacked_dir = dirs["dataprep"] / "stacked"
stacked_dir.mkdir(exist_ok=True)

sys.argv = [
    "trazo-pt2-dataprep",
    "--window-a-dir", str(dirs["chips"] / "window_a"),
    "--window-b-dir", str(dirs["chips"] / "window_b"),
    "--out-dir",      str(stacked_dir),
]

from trazo.pt2_dataprep import pair_stacks
pair_stacks.main(sys.argv[1:])

---
## Step 2b — Resize Chips to 256×256

Standardizes every 8-band stack to exactly 256×256 pixels (crops, pads, or inpaints as needed). Outputs go into a `sized256/` subfolder.

**CLI equivalent:**
```bash
trazo-pt2-dataprep resize-256 \
  --base-folder ./trazo_workspace/03_dataprep/stacked \
  --bands-required 8 \
  --overwrite
```

In [ ]:
sys.argv = [
    "trazo-pt2-dataprep",
    "--base-folder",    str(stacked_dir),
    "--bands-required", "8",
    "--overwrite",
]

from trazo.pt2_dataprep import resize_chips_256
resize_chips_256.main(sys.argv[1:])

---
## Step 2c — Create Masks

Splits 8-band chips into `window_a` / `window_b` halves and rasterizes field polygons into instance and semantic masks.

**CLI equivalent:**
```bash
trazo-pt2-dataprep make-masks \
  --base-folder ./trazo_workspace/03_dataprep/stacked \
  --fields-shp  ./my_fields.geojson \
  --overwrite
```

In [ ]:
sys.argv = [
    "trazo-pt2-dataprep",
    "--base-folder", str(stacked_dir),
    "--fields-shp",  str(FIELDS_FILE),
    "--overwrite",
]

from trazo.pt2_dataprep import make_masks_and_windows
make_masks_and_windows.main(sys.argv[1:])

---
## Step 2d — Export to .hkl (for pt4 training)

Packages `window_a`, `window_b`, and masks into HDF5/hickle `.hkl` files — the format consumed by the pt4 `FTW` dataset loader.

**CLI equivalent:**
```bash
trazo-pt2-dataprep export-hkl \
  --root       ./trazo_workspace/03_dataprep \
  --mask-type  semantic_3class \
  --overwrite
```

In [ ]:
sys.argv = [
    "trazo-pt2-dataprep",
    "--root",      str(dirs["dataprep"]),
    "--mask-type", "semantic_3class",
    "--overwrite",
]

from trazo.pt2_dataprep import export_hkl
export_hkl.main(sys.argv[1:])

In [ ]:
# Visualize a chip + its mask
import hickle as hkl
import matplotlib.pyplot as plt
import numpy as np

sample_hkl = sorted(dirs["dataprep"].rglob("*.hkl"))
if sample_hkl:
    data = hkl.load(str(sample_hkl[0]))
    print("hkl keys:", list(data.keys()) if isinstance(data, dict) else type(data))

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    # Assumes data is (window_a, window_b, mask) or similar — adjust to your schema
    for ax, arr, title in zip(axes, data if isinstance(data, (list, tuple)) else [data], 
                               ["Window A", "Window B", "Mask"]):
        if hasattr(arr, 'shape'):
            rgb = arr[:3].transpose(1,2,0) if arr.ndim == 3 else arr
            rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
            ax.imshow(rgb)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No .hkl files found yet — run export-hkl above.")

---
## Step 3 - Fine-tune an existing checkpoint *(pt3)*

Adapt a pretrained model (for example the FTW 3-class checkpoint) to your region.

Four strategies, with the technical note's Chiquitania results. The FTW baseline
scored 42.5% pixel IoU on that test set.

| strategy | updates | pixel IoU | notes |
|---|---|---|---|
| `full` | every weight | 84.4% | best on target, forgets the most |
| `lastlayer` | segmentation head | 62.7% | cheapest, underfits |
| `lora` | rank-r adapters | 64.0% | small trainable footprint |
| `upgd` | every weight, utility-gated | 65.6% | some forgetting protection |

Each strategy has a bundled starter config, so `--config` is optional.

**CLI equivalent:**
```bash
trazo-pt3-finetune run --strategy full --checkpoint model.ckpt \
  --data-dir ./dataprep --output-dir ./ft
```

In [ ]:
# Fine-tune. Omit --config to use the bundled config for the strategy.
from trazo.pt3_finetune.cli import main as pt3_main

STRATEGY = "full"   # full | lastlayer | lora | upgd

pt3_main([
    "run",
    "--strategy",   STRATEGY,
    "--checkpoint", str(CHECKPOINT),
    "--data-dir",   str(dirs["dataprep"]),
    "--output-dir", str(dirs["checkpoints"] / f"ft_{STRATEGY}"),
])

---
## Step 3b - Merge to limit catastrophic forgetting

Full fine-tuning on a small regional sample is the strongest option on the target
region and the worst everywhere else: in the note, India pixel IoU fell from 96.0%
to 0.1%. MagMax merges the fine-tuned weights back toward the base model, keeping
the larger-magnitude change per weight. That recovered Kenya to 65.9% and India to
43.4% while still holding 79.2% on the target region.

In [ ]:
# Merge the fine-tuned checkpoint back toward the base model
from trazo.pt3_finetune.cli import main as pt3_main

ft_ckpts = sorted((dirs["checkpoints"] / f"ft_{STRATEGY}").rglob("*.ckpt"))
print("Fine-tuned checkpoints:", [str(c) for c in ft_ckpts])

if ft_ckpts:
    pt3_main([
        "merge",
        "--base",         str(CHECKPOINT),
        "--finetuned",    str(ft_ckpts[-1]),
        "--output",       str(dirs["checkpoints"] / "merged.ckpt"),
        "--method",       "magmax",
        "--scaling-coef", "1.0",
    ])

---
## Step 4 — Train *(pt4)* — requires `pt4` extras

Full PyTorch Lightning training run using `LightningCLI`. Reads the `.hkl` files produced in Step 2d.

**CLI equivalent:**
```bash
trazo-pt4-train fit \
  --config     ./configs/train_config.yaml \
  --data-dir   ./trazo_workspace/03_dataprep \
  --output-dir ./trazo_workspace/04_checkpoints
```

In [ ]:
# Training is best run as a subprocess to avoid notebook / CUDA state issues
import subprocess, sys
from pathlib import Path
import trazo

# A working example config ships with the package
EXAMPLE_CONFIG = Path(trazo.__file__).parent / "pt4_train" / "configs" / "example_3class.yaml"
config = TRAIN_CONFIG if TRAIN_CONFIG.exists() else EXAMPLE_CONFIG
print("Using config:", config)

cmd = [
    sys.executable, "-m", "trazo.pt4_train.cli",
    "fit",
    "--config",     str(config),
    "--data-dir",   str(dirs["dataprep"]),      # overrides data.init_args.root
    "--output-dir", str(dirs["checkpoints"]),   # overrides trainer.default_root_dir
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=False, text=True)
print("Return code:", result.returncode)

In [ ]:
# Find the best checkpoint after training
checkpoints = sorted(dirs["checkpoints"].rglob("*.ckpt"))
print("Checkpoints found:")
for c in checkpoints:
    print(" ", c)

---
## Step 5 — Inference *(pt5)* — requires `pt5` / `active` extras

Run field boundary detection over new Sentinel-2 imagery.

**CLI equivalent:**
```bash
trazo-pt5-infer multi-infer \
  --model      ./trazo_workspace/04_checkpoints/best.ckpt \
  --chips-dir  ./trazo_workspace/03_dataprep/stacked/sized256 \
  --out-dir    ./trazo_workspace/05_inference
```

In [ ]:
# Point to the best checkpoint
best_ckpt = checkpoints[-1] if checkpoints else CHECKPOINT
print("Using checkpoint:", best_ckpt)

sized256_dir = stacked_dir / "sized256"

# batch-infer runs one checkpoint over a folder of stacks.
# Use multi-infer instead when comparing several checkpoints.
sys.argv = [
    "trazo-pt5-infer",
    "--input-dir",        str(sized256_dir),
    "--model-checkpoint", str(best_ckpt),
    "--output-dir",       str(dirs["inference"]),
]

from trazo.pt5_inference import batchinference
batchinference.main(sys.argv[1:])

In [ ]:
# Visualize a prediction
import rasterio
import matplotlib.pyplot as plt
import numpy as np

pred_files = sorted(dirs["inference"].rglob("*.tif"))
print(f"Inference outputs: {len(pred_files)} TIFs")

if pred_files:
    with rasterio.open(pred_files[0]) as src:
        pred = src.read(1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    # Load corresponding input chip if available
    chip_path = sized256_dir / pred_files[0].name
    if chip_path.exists():
        with rasterio.open(chip_path) as src:
            rgb = src.read([1, 2, 3]).transpose(1, 2, 0).astype(float)
            rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
        ax1.imshow(rgb)
        ax1.set_title("Input (RGB bands 1-3)")
    else:
        ax1.set_title("(input chip not found)")
    ax1.axis('off')

    ax2.imshow(pred, cmap="tab10", vmin=0, vmax=2)
    ax2.set_title("Prediction\n0=background  1=interior  2=boundary")
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

---
## Active Learning Sampling *(optional)*

After inference, use active learning to select the most informative chips for re-labeling.

**CLI equivalent:**
```bash
trazo-active-sample \
  --chips-dir  ./trazo_workspace/03_dataprep/stacked/sized256 \
  --checkpoint ./trazo_workspace/04_checkpoints/best.ckpt \
  --ag-raster  ./ag_binary.tif \
  --output-dir ./trazo_workspace/active_learning \
  --n-per-strategy 50
```

In [ ]:
AG_RASTER = Path("./ag_binary.tif").resolve()   # binary ag land-cover raster
active_dir = WORKSPACE / "active_learning"
active_dir.mkdir(exist_ok=True)

sys.argv = [
    "trazo-active-sample",
    "--chips-dir",       str(sized256_dir),
    "--checkpoint",      str(best_ckpt),
    "--ag-raster",       str(AG_RASTER),
    "--output-dir",      str(active_dir),
    "--n-per-strategy",  "50",
]

from trazo.pt1_createdata.active_sample import main as active_main
active_main()

---
## Quick reference: all CLI commands

```bash
# Install check
trazo-smoke
trazo-smoke --work-dir ./smoke   # keep the outputs

# Step 1
trazo-pt1-create --help
trazo-pt1-create grid --help
trazo-pt1-create chips --help

# Step 2
trazo-pt2-dataprep --help
trazo-pt2-dataprep pair-stacks --help    # --window-a-dir --window-b-dir --out-dir
trazo-pt2-dataprep resize-256 --help     # --base-folder
trazo-pt2-dataprep chips-bboxes --help   # --folder
trazo-pt2-dataprep make-masks --help     # --base-folder --fields-shp
trazo-pt2-dataprep chips-parquet --help  # --base-folder --fields-shp
trazo-pt2-dataprep scale-u16 --help      # --base-folder
trazo-pt2-dataprep export-hkl --help     # --root
trazo-pt2-dataprep export-zarr --help    # --root

# Step 3
trazo-pt3-finetune --help
trazo-pt3-finetune configs               # where the bundled configs live
trazo-pt3-finetune run --help            # --strategy full|lastlayer|lora|upgd
trazo-pt3-finetune merge --help          # MagMax / task-vector merging

# Step 4
trazo-pt4-train --help
trazo-pt4-train fit --help
trazo-pt4-train test --help

# Step 5
trazo-pt5-infer --help
trazo-pt5-infer tilepairs --help
trazo-pt5-infer tilepairs-tilelist --help
trazo-pt5-infer multi-infer --help       # many checkpoints
trazo-pt5-infer batch-infer --help       # one checkpoint

# Active learning
trazo-active-sample --help
```